In [2]:
import requests
import pandas as pd

In [3]:
# dictionary of all teams:
teams = requests.get('https://statsapi.mlb.com/api/v1/teams', params={'division' : 'Major League Baseball'}).json()

In [18]:
# hardcoded api keys:
intl_league_id = 117
pcl_league_id = 112
aaa_ids = [117, 112]

eastern_league_id = 113
southern_league_id = 111
texas_league_id = 109
aa_ids = [113, 111, 109]

midwest_league_id = 118
sally_league_id = 116 # id 148 returns a DIFFERENT south atlantic league (most recent season 1963)
northwest_league_id = 126
high_a_ids = [118, 116, 126]

carolina_league_id = 122
florida_state_league_id = 123
california_league_id = 110
low_a_ids = [122, 123, 110]

az_complex_league_id = 121
fl_complex_league_id = 123
dsl_league_id = 130
rookie_ids = [121, 123, 130]

In [5]:
def get_league_team_ids(team_dict, league_id):
    print(f'Fetching list of teams in league {league_id}:')
    print('----------------------------------------------')
    team_id_list = []
    for team in team_dict['teams']:
        if 'id' in team['league'] and team['league']['id'] == league_id:
            print(team['name'])
            team_id_list.append(team['id'])
    return team_id_list

def get_team_roster_ids(team_id):
    roster = requests.get(f'https://statsapi.mlb.com/api/v1/teams/{team_id}/roster').json()
    player_id_list = []
    #print(roster['roster'])
    for player in roster['roster']:
        print(f'{player['person']['id']}  {player['person']['fullName']}')
        player_id_list.append(player['person']['id'])
    return player_id_list

def get_minor_logs(player_id, season):
    return requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats?stats=gameLog&gameType=R&leagueListId=milb_all&group=hitting&hydrate=team(league)&language=en&season={season}",).json()

def get_major_logs(player_id, season):
    return requests.get(f'https://statsapi.mlb.com/api/v1/people/{player_id}/stats?season={season}&group=hitting&stats=gameLog',).json()

def tabulate(log_file):
    pass
    # Hardcoded dictionary of positions?
    # position_list_raw_tallies = {'P': 0, etc.}
    # position_list_percent = {'P': None, etc.} # Cuz maybe I want an undefined flag for levels that they didn't play at?
    # Do I define these for every level? Or do I wait and call them into existence when they're needed? that's probably better
    # I need to figure out how best to stack this data into a tensor. That's the key design decision here.
    # A, AA, AAA, MLB, etc = []
    #for game in log_file[]: # This is a placeholder
        # Task 1: identify level (because the minor league logs are mixed I believe)
        # This code may have to handle MLB logs different from milb because they are differently structured
        # When a level comes up, add a tally to that count, then inspect the position, and add tallies as appropriate
    # Then, outside of loop, run a function against the raw position tallies to convert them to percentages for analysis
    # Thinking further, I wanted to exclude PH (and potentially pinch runner and other such things if they're recorded) but doing so may
    # cause a divide by zero in the edgecase where that's the only appearance at that level. And I DO want to record the appearance!
    # Ultimately, this:
    # return [data package that can slide into the tensor's seasonal columns]
    # The more I think about this the more I think I want to precollate the MLB and milb logs before they enter this function, would be so much cleaner

In [16]:
def game_count_by_level(minor_logs, major_logs):
    r = 0
    low_a = 0
    hi_a = 0
    aa = 0
    aaa = 0
    if len(minor_logs['stats']) != 0:
        for game in minor_logs['stats'][0]['splits']:
            if game['league']['id'] in aaa_ids:
                aaa += 1
            elif game['league']['id'] in aa_ids:
                aa += 1
            elif game['league']['id'] in high_a_ids:
                hi_a += 1
            elif game['league']['id'] in low_a_ids:
                low_a += 1
            elif game['league']['id'] in rookie_ids:
                r += 1
    if len(major_logs['stats']) != 0:
        mlb = len(major_logs['stats'][0]['splits'])
    else:
        mlb = 0
    return [r, low_a, hi_a, aa, aaa, mlb] # Returning as list for now; may need to change structure

In [19]:
# Testing game_count_by_level using Blake Sabol's 2025. Expecting 8 MLB appearances, 66 triple A, else all zeros
sabol2025_minors = get_minor_logs(666165, 2025)
sabol2025_majors = get_major_logs(666165, 2025)
sabol2025_vector = game_count_by_level(sabol2025_minors, sabol2025_majors)
sabol2025_vector

[0, 0, 0, 0, 66, 8]

In [20]:
def get_all_season_vectors(player_id, season_start, season_stop): # start and stop are inclusive
    vector_list = []
    for season in range(season_start, season_stop + 1):
        player_minors = get_minor_logs(player_id, season)
        player_majors = get_major_logs(player_id, season)
        season_vec = game_count_by_level(player_minors, player_majors)
        vector_list.append(season_vec) # maintains each season as a separately nested vector in list
    return vector_list

In [21]:
# Testing get_all_season_vectors with Blake Sabol 2021-2026
sabol_vector_list = get_all_season_vectors(666165, 2021, 2026)
sabol_vector_list

[[0, 14, 52, 0, 0, 0],
 [0, 0, 0, 98, 25, 0],
 [0, 0, 0, 0, 0, 110],
 [0, 0, 0, 3, 97, 11],
 [0, 0, 0, 0, 66, 8],
 [0, 0, 0, 0, 101, 0]]

I confirmed via b-ref that the above vector list is accurate for Sabol's 2021-2026.

In [6]:
intl_league_team_ids = get_league_team_ids(teams, intl_league_id)
print(intl_league_team_ids)

Fetching list of teams in league 117:
----------------------------------------------
Durham Bulls
Memphis Redbirds
Louisville Bats
Buffalo Bisons
Gwinnett Stripers
Columbus Clippers
Iowa Cubs
Indianapolis Indians
Charlotte Knights
Toledo Mud Hens
Scranton/Wilkes-Barre RailRiders
Worcester Red Sox
Rochester Red Wings
Omaha Storm Chasers
Syracuse Mets
Nashville Sounds
Jacksonville Jumbo Shrimp
Norfolk Tides
Lehigh Valley IronPigs
St. Paul Saints
[234, 235, 416, 422, 431, 445, 451, 484, 494, 512, 531, 533, 534, 541, 552, 556, 564, 568, 1410, 1960]


In [13]:
intl_league_rosters = []
for team in intl_league_team_ids:
    team_roster = get_team_roster_ids(team)
    intl_league_rosters.extend(team_roster)
#intl_league_rosters

693739  Alexander Alberto
596103  Austin Slater
666165  Blake Sabol
805207  Caden Bodine
657611  Cam Hill
805363  Chris Clark
682126  Evan Reifert
683344  Homer Bush Jr.
669261  Jack Suwinski
666931  Jared Oliva
680701  John Rooney
687866  Justin Yeager
676831  Kodi Whitley
669722  Logan Davidson
694312  Logan Workman
687189  Marcus Johnson
686554  Oliver Dunn
801643  Owen Wild
694420  Tatem Levins
691003  Tre' Morgan
800060  Xavier Isaac
669697  Andrew Schultz
647378  Andy Yerzy
676153  Austin Love
676632  Bligh Madris
822410  Bobby Olsen
825576  Brandt Thompson
690914  Brody Moore
690971  Chase Davis
807742  Colton Ledbetter
703766  Dakota Harris
815104  Graysen Tarlow
676596  Josh Fleming
701388  Jurrangelo Cijntje
695766  Mason Molina
702331  Noah Mendlinger
677337  Pete Hansen
680636  Ramon Mendoza
694885  Ryan Murphy
811139  Sandro Santana
813766  Skylar Hales
813852  Trey Paige
804241  Zach Levenson
664948  Anthony Misiewicz
686844  Ben Wereski
605335  Brandon Leibrandt
694650  

In [12]:
len(intl_league_rosters)

464

I'll make a nested list of each player's season vectors from 2021-2026. I'll call it master_list. The details are:
- Sample is all players listed on AAA rosters at time of code execution, which = `len(intl_league_rosters)`
- Each player is a row; that player's row contains their season vector list for 2021-2026

In [24]:
master_list = []
for player_id in intl_league_rosters:
    master_list.append(get_all_season_vectors(player_id, 2021, 2026))
master_list


[[[0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 129],
  [0, 0, 0, 0, 0, 125],
  [0, 0, 0, 0, 8, 89],
  [0, 0, 0, 0, 5, 84],
  [0, 0, 0, 0, 9, 65],
  [0, 0, 0, 0, 45, 28]],
 [[0, 14, 52, 0, 0, 0],
  [0, 0, 0, 98, 25, 0],
  [0, 0, 0, 0, 0, 110],
  [0, 0, 0, 3, 97, 11],
  [0, 0, 0, 0, 66, 8],
  [0, 0, 0, 0, 101, 0]],
 [[0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 11, 0, 0, 0, 0],
  [0, 33, 32, 37, 15, 0]],
 [[0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0],
